# Overfitting / generalization checks for verifier-CE

Two questions the in-distribution result cannot answer on its own:

**Check 1 — train-test gap (example memorization).** Score a CE checkpoint
on its OWN training set and compare to its held-out accuracy. A small gap
means it generalized within the distribution; a large gap means it
memorized examples.

**Check 2 — out-of-distribution (did it learn the RULE or the generator?).**
Train CE on 3 domains (email, payment, repo), test on 2 unseen domains
(file, db) whose actions and resource formats never appeared in training.
High OOD accuracy = learned the authorization rule; a sharp drop = learned
the distribution. (budget_violation is payment-only, so the OOD test covers
8 of 9 classes — disclose this.)

Runtime → T4/L4 GPU → Run all. Download `overfitting_checks.zip` at the end.

In [ ]:
!git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
%cd verifier-as-reward
import torch; assert torch.cuda.is_available()
!PYTHONPATH=. python test_authority_verifier.py

## Check 1 — train-test gap (retrains CE quickly, then evals on train vs val vs committed test)

In [ ]:
# Rebuild the in-distribution corpora, CE-train seed 7 (single seed is enough
# for a gap measurement), keep the checkpoint fresh for evaluation.
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 \
  --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25
!PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
  --ce-loss --balance-reward --prompt-style nl \
  --steps 500 --batch-size 16 --lr 2e-5 \
  --train-file expanded_train.jsonl --test-file expanded_val.jsonl \
  --eval-every 50 --eval-max-actions 120 --seed 7 \
  --log-file training_log_ce_gap_seed7.jsonl --save-dir ckpt_ce_gap_seed7

In [ ]:
# Same checkpoint scored on TRAIN, VAL, and the committed TEST (NL prompts).
import json
json.dump({"backends": {}}, open("results_gap.json", "w"))
for name, f in [("train", "expanded_train.jsonl"),
                ("val", "expanded_val.jsonl"),
                ("committed_test", "benchmark_test.jsonl")]:
    print(f"\n### checkpoint on {name} ({f}) ###")
    !PYTHONPATH=. python train_verifier_reward.py \
      --eval-checkpoint ckpt_ce_gap_seed7 --test-file {f} \
      --merge-results results_gap_{name}.json

## Check 2 — out-of-distribution domain hold-out

In [ ]:
# Train on email/payment/repo, hold out file/db entirely.
!PYTHONPATH=. python make_ood_split.py --seed 303 --traces-per-class 200 \
  --train-domains email,payment,repo --test-domains file,db
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
    --ce-loss --balance-reward --prompt-style nl \
    --steps 500 --batch-size 16 --lr 2e-5 \
    --train-file ood_train.jsonl --test-file ood_test.jsonl \
    --eval-every 50 --eval-max-actions 200 --seed $s \
    --log-file training_log_ood_seed$s.jsonl --save-dir ckpt_ood_seed$s || break ; done

In [ ]:
# OOD checkpoints scored on the held-out-domain test set (all 1150 actions).
import json
json.dump({"backends": {}}, open("results_ood.json", "w"))
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py \
    --eval-checkpoint ckpt_ood_seed$s --test-file ood_test.jsonl \
    --merge-results results_ood.json || break ; done
print(json.dumps(json.load(open("results_ood.json")), indent=2)[:1200])

In [ ]:
!zip -j overfitting_checks.zip training_log_ce_gap_seed7.jsonl \
  training_log_ood_seed*.jsonl results_gap_*.json results_ood.json
from google.colab import files
files.download("overfitting_checks.zip")